# MindRead -- GRPO Training on Kaggle T4
Run all cells top to bottom. Takes ~25 min total. GPU must be ON (T4 x1).

In [ ]:
# CELL 1 -- Install deps
# System TRL is 0.11.4 which has no GRPO (added in 0.13+).
# Install latest TRL to /tmp/trl_new and prepend to sys.path.
import subprocess, sys, os

TARGET = '/tmp/trl_new'
os.makedirs(TARGET, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--target', TARGET, 'trl'], check=True)

for key in list(sys.modules.keys()):
    if key.startswith('trl'):
        del sys.modules[key]

sys.path.insert(0, TARGET)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn[standard]', 'sentence-transformers>=3.3.0',
    'groq', 'python-dotenv', 'httpx', 'pydantic==2.9.2',
    'datasets==3.0.1', 'rich', 'scikit-learn>=1.6.0',
    'transformers>=4.45.2', 'accelerate>=0.34.2'], check=True)

from trl import GRPOConfig, GRPOTrainer
import trl
print('Done. TRL version:', trl.__version__)
print('GRPOConfig OK')

In [ ]:
# CELL 2 -- Set your Groq API key
import os
os.environ['GROQ_API_KEY'] = 'PASTE_YOUR_GROQ_KEY_HERE'
print('Key set:', os.environ['GROQ_API_KEY'][:12] + '...')

In [ ]:
# CELL 3 -- Copy project from Kaggle Dataset
import shutil, os

WORK_DIR = '/kaggle/working/mindread-env'

CANDIDATE_PATHS = [
    '/kaggle/input/mindread-env',
    '/kaggle/input/datasets/mr6436/mindread-env/mindread',
    '/kaggle/input/mindread-env/mindread',
    '/kaggle/input/datasets/mr6436/mindread/mindread',
]

DATASET_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p) and os.path.exists(os.path.join(p, 'server', 'main.py')):
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'main.py' in files and os.path.basename(root) == 'server':
            DATASET_PATH = os.path.dirname(root)
            break

if DATASET_PATH:
    shutil.copytree(DATASET_PATH, WORK_DIR, dirs_exist_ok=True)
    print('Copied from:', DATASET_PATH)
else:
    raise RuntimeError('Dataset not found -- add the mindread dataset to this notebook')

os.chdir(WORK_DIR)
import sys
sys.path.insert(0, WORK_DIR)
print('Working dir:', os.getcwd())
print('server/main.py exists:', os.path.exists('server/main.py'))

In [ ]:
# CELL 4 -- Start environment server (threading so GROQ_API_KEY is inherited)
import threading, time, httpx, os

def run_server():
    import uvicorn
    uvicorn.run('server.main:app', host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()

for i in range(30):
    time.sleep(2)
    try:
        r = httpx.get('http://localhost:8000/health', timeout=3)
        if r.status_code == 200:
            print(f'Server up after {(i+1)*2}s:', r.json())
            break
    except:
        print(f'  waiting... ({(i+1)*2}s)')
else:
    print('Server failed to start -- check GROQ_API_KEY is set in Cell 2')

In [ ]:
# CELL 5 -- Verify all 5 tasks
import httpx
r = httpx.get('http://localhost:8000/tasks')
tasks = r.json()
for t in tasks:
    print(f"  {t['id']} | max_steps={t['max_steps']} | {t['difficulty']}")
print('\nAll tasks OK' if len(tasks) == 5 else 'ERROR: missing tasks')

In [ ]:
# CELL 6 -- Sanity check: one full episode
import httpx, json

client = httpx.Client(base_url='http://localhost:8000', timeout=60)

obs = client.post('/reset', json={'task_id': 'factual_easy', 'secret_id': 'fe_001'}).json()
ep_id = obs['episode_id']
print('Episode:', ep_id)
print('Oracle:', obs['oracle_persona'])

step = client.post('/step', json={
    'episode_id': ep_id,
    'action': {'action': 'ask_question', 'question': 'What projects are you most focused on?'}
}).json()
print('Oracle says:', step['info']['oracle_response'])

result = client.post('/submit', json={
    'episode_id': ep_id,
    'hypothesis': 'The Q3 product launch was postponed due to a compliance issue.',
    'category_prediction': 'factual'
}).json()
print(f"\nReward: {result['reward']}")
print(f"True secret: {result['true_secret']}")
print('\nSanity check PASSED' if result['reward'] >= 0 else 'ERROR')

In [ ]:
# CELL 7 -- Load model (Qwen2.5-1.5B fits on T4 in bfloat16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Loading {MODEL}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)
print('Model loaded. Params:', sum(p.numel() for p in model.parameters()) / 1e9, 'B')

In [ ]:
# CELL 8 -- Build prompt dataset (100 episodes of factual_easy)
import json, httpx
from datasets import Dataset
from training.mindread_grpo_env import MindReadGRPOEnv

env = MindReadGRPOEnv(base_url='http://localhost:8000')

TASK_ID = 'factual_easy'
N_EPISODES = 100

print(f'Building {N_EPISODES} prompts for task={TASK_ID}...')
prompts, metas = [], []

for i in range(N_EPISODES):
    try:
        obs = env.reset(task_id=TASK_ID)
        system, user = env.build_prompt(obs)
        prompt = f'<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n'
        prompts.append({'prompt': prompt})
        metas.append(json.dumps({'episode_id': obs['episode_id'], 'obs': obs}))
        if (i+1) % 10 == 0:
            print(f'  {i+1}/{N_EPISODES}')
    except Exception as e:
        print(f'  [warn] episode {i} failed: {e}')

dataset = Dataset.from_list(prompts)
dataset = dataset.add_column('episode_meta', metas)
print(f'Dataset ready: {len(dataset)} episodes')

In [ ]:
# CELL 9 -- Define reward function
import json
from training.mindread_grpo_env import MindReadGRPOEnv

env = MindReadGRPOEnv(base_url='http://localhost:8000')
reward_log = []

def mindread_reward(completions, episode_meta, **kwargs):
    rewards = []
    for completion, meta_str in zip(completions, episode_meta):
        meta = json.loads(meta_str)
        try:
            result = env.evaluate_completion(meta['episode_id'], completion, meta['obs'])
            rewards.append(result.reward)
            reward_log.append({
                'reward': result.reward,
                'questions': result.questions_asked,
                'breakdown': result.breakdown
            })
        except Exception as e:
            rewards.append(0.0)
    avg = sum(rewards) / len(rewards) if rewards else 0
    print(f'  batch rewards: {[round(r,3) for r in rewards]} | avg={avg:.3f}')
    return rewards

print('Reward function ready')

In [ ]:
# CELL 10 -- GRPO Training (300 steps, ~20 min on T4)
import sys, torch

if '/tmp/trl_new' not in sys.path:
    sys.path.insert(0, '/tmp/trl_new')

from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='mindread-detective-v1',
    learning_rate=1e-5,
    max_steps=300,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_completion_length=512,
    bf16=True,
    logging_steps=5,
    save_steps=100,
    report_to=[],
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=mindread_reward,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print('Starting GRPO training for 300 steps...')
print('Watch the reward numbers climb and questions_asked drop.')
trainer.train()

In [ ]:
# CELL 11 -- Show training curve
import statistics

if reward_log:
    rewards_all = [r['reward'] for r in reward_log]
    questions_all = [r['questions'] for r in reward_log]

    n = len(rewards_all)
    window = max(1, n // 6)
    print('\n=== Training Curve ===')
    print(f'{"Window":<12} {"Avg Reward":<14} {"Avg Questions"}')
    print('-' * 40)
    for i in range(0, n, window):
        chunk_r = rewards_all[i:i+window]
        chunk_q = questions_all[i:i+window]
        label = f'steps {i}-{min(i+window, n)}'
        avg_r = statistics.mean(chunk_r)
        avg_q = statistics.mean(chunk_q)
        print(f'{label:<12} {avg_r:<14.4f} {avg_q:.1f}')

    print(f'\nFINAL avg reward:    {statistics.mean(rewards_all[-50:]):.4f}')
    print(f'FINAL avg questions: {statistics.mean(questions_all[-50:]):.1f}')
    print(f'BASELINE avg reward:    ~0.14')
    print(f'BASELINE avg questions: ~7.7')
else:
    print('No reward log yet -- run Cell 10 first')

In [ ]:
# CELL 12 -- Save results to evals/trained_results.md
import statistics, os
from datetime import datetime

if reward_log:
    final_50 = reward_log[-50:]
    avg_r = statistics.mean(r['reward'] for r in final_50)
    avg_q = statistics.mean(r['questions'] for r in final_50)

    rewards_all = [r['reward'] for r in reward_log]
    questions_all = [r['questions'] for r in reward_log]
    n = len(rewards_all)
    window = max(1, n // 6)

    curve = ''
    for i in range(0, n, window):
        chunk_r = rewards_all[i:i+window]
        chunk_q = questions_all[i:i+window]
        curve += f'steps {i}-{min(i+window,n)}: reward={statistics.mean(chunk_r):.4f} questions={statistics.mean(chunk_q):.1f}\n'

    content = f"""# MindRead Evaluation Results -- Post-GRPO Training
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Training: 300 steps, Qwen2.5-1.5B-Instruct, GRPO via TRL
Task: factual_easy

| Task | Avg Reward | Avg Questions |
|------|-----------|---------------|
| factual_easy | {avg_r:.4f} | {avg_q:.1f} |

## vs Baseline

| Metric | Baseline | Trained | Improvement |
|--------|---------|---------|-------------|
| Avg Reward | 0.1393 | {avg_r:.4f} | +{((avg_r - 0.1393) / 0.1393 * 100):.0f}% |
| Avg Questions | 7.7 | {avg_q:.1f} | {((7.7 - avg_q) / 7.7 * 100):.0f}% fewer |

## Training Curve
{curve}"""

    os.makedirs('evals', exist_ok=True)
    with open('evals/trained_results.md', 'w') as f:
        f.write(content)
    print('Written to evals/trained_results.md')
    print(content)
else:
    print('No reward log -- run Cell 10 first')

In [ ]:
# CELL 13 -- Download trained_results.md
from IPython.display import FileLink
FileLink('evals/trained_results.md')